## Retailpulse Gold Aggregation

#### Configuration

This cell sets up the source and target schema variables for the Retailpulse Gold Aggregation process.  
- `SILVER_SCHEMA`: Source schema for silver-level data (`workspace.retailpulse_silver`)
- `GOLD_SCHEMA`: Target schema for gold-level aggregated data (`workspace.retailpulse_gold`)

It also prints the loaded configuration for verification.

In [0]:
# Cell 2 - Configuration
SILVER_SCHEMA = "workspace.retailpulse_silver"
GOLD_SCHEMA = "workspace.retailpulse_gold"

print("Configuration loaded.")
print(f"Source: {SILVER_SCHEMA}")
print(f"Target: {GOLD_SCHEMA}")

Configuration loaded.
Source: workspace.retailpulse_silver
Target: workspace.retailpulse_gold


#### Load Silver Tables

This cell loads the main silver-level tables required for aggregation:

- **orders**: Transactional order data
- **customers**: Customer information
- **products**: Product details

It also prints the row counts for each table to verify successful loading.

In [0]:
# Cell 4 - Load Silver tables
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round

df_orders = spark.table(f"{SILVER_SCHEMA}.orders")
df_customers = spark.table(f"{SILVER_SCHEMA}.customers")
df_products = spark.table(f"{SILVER_SCHEMA}.products")

print(f"Orders: {df_orders.count():,} rows")
print(f"Customers: {df_customers.count():,} rows")
print(f"Products: {df_products.count():,} rows")

Orders: 500,000 rows
Customers: 50,000 rows
Products: 5,000 rows


#### Gold Aggregation: Daily Sales

This cell creates the `daily_sales` gold-level aggregate table:

- Groups order data by `order_date`
- Calculates:
  - `total_orders`: Number of orders per day
  - `total_revenue`: Total revenue per day (rounded to 2 decimals)
- Orders results by `order_date`
- Displays the row count and a sample of the aggregated data

In [0]:
# Cell 6 - Gold: daily_sales
df_daily_sales = (
    df_orders
    .groupBy("order_date")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("amount"), 2).alias("total_revenue")
    )
    .orderBy("order_date")
)

print(f"Daily sales rows: {df_daily_sales.count():,}")
df_daily_sales.show(5)

Daily sales rows: 1,827
+----------+------------+-------------+
|order_date|total_orders|total_revenue|
+----------+------------+-------------+
|2020-01-01|         276|    411922.71|
|2020-01-02|         253|    412737.05|
|2020-01-03|         302|    557098.34|
|2020-01-04|         268|    375960.57|
|2020-01-05|         256|    336627.00|
+----------+------------+-------------+
only showing top 5 rows


#### Gold Aggregation: Top Products

This cell creates the `top_products` gold-level aggregate table:

- Groups order data by `product_id`
- Calculates:
  - `total_orders`: Number of orders per product
  - `total_revenue`: Total revenue per product (rounded to 2 decimals)
- Joins with the products table to include product details (`name`, `category`)
- Orders results by `total_revenue` in descending order
- Displays the row count and a sample of the aggregated data

In [0]:
# Cell 8 - Gold: top_customers
df_top_customers = (
    df_orders
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("amount"), 2).alias("lifetime_value")
    )
    .join(df_customers.select("customer_id", "name", "city", "country"), "customer_id")
    .orderBy(col("lifetime_value").desc())
)

print(f"Top customers rows: {df_top_customers.count():,}")
df_top_customers.show(5)

Top customers rows: 50,000
+-----------+------------+--------------+----------------+----------+-------+
|customer_id|total_orders|lifetime_value|            name|      city|country|
+-----------+------------+--------------+----------------+----------+-------+
|       8283|          17|      94762.70|     Lucas Allen| Vancouver|     CA|
|      29629|          10|      77858.18|     Lucas Lewis|   Houston|     US|
|      30365|          19|      72471.26|    Elijah Perez|Birmingham|     GB|
|      38007|          16|      71575.79|Sophia Rodriguez|    Mumbai|     IN|
|      47140|          11|      70916.10|      Sophia Lee|    Sydney|     AU|
+-----------+------------+--------------+----------------+----------+-------+
only showing top 5 rows



#### Gold Aggregation: Top Customers

This cell creates the `top_customers` gold-level aggregate table:

- Groups order data by `customer_id`
- Calculates:
  - `total_orders`: Number of orders per customer
  - `lifetime_value`: Total revenue per customer (rounded to 2 decimals)
- Joins with the customers table to include customer details (`name`, `city`, `country`)
- Orders results by `lifetime_value` in descending order
- Displays the row count and a sample of the aggregated data

In [0]:
# Cell 10 - Gold: revenue_by_category
df_revenue_by_category = (
    df_orders
    .join(df_products.select("product_id", "category"), "product_id")
    .groupBy("category")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("amount"), 2).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

print(f"Revenue by category rows: {df_revenue_by_category.count():,}")
df_revenue_by_category.show()

Revenue by category rows: 6
+--------------+------------+-------------+
|      category|total_orders|total_revenue|
+--------------+------------+-------------+
|   Electronics|       79822| 443995204.50|
|Home & Kitchen|       85191| 122109022.14|
|        Sports|       84508|  94486921.14|
|      Clothing|       83165|  73313246.05|
|        Beauty|       84870|  35868988.35|
|         Books|       82444|  19423098.88|
+--------------+------------+-------------+




#### Gold Aggregation: Orders by Country

This cell creates the `orders_by_country` gold-level aggregate table:

- Joins order data with the customers table to associate each order with a country
- Groups data by `country`
- Calculates:
  - `total_orders`: Number of orders per country
  - `total_revenue`: Total revenue per country (rounded to 2 decimals)
- Orders results by `total_revenue` in descending order
- Displays the row count and a sample of the aggregated data

In [0]:
# Cell 12 - Gold: orders_by_country
df_orders_by_country = (
    df_orders
    .join(df_customers.select("customer_id", "country"), "customer_id")
    .groupBy("country")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("amount"), 2).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

print(f"Orders by country rows: {df_orders_by_country.count():,}")
df_orders_by_country.show()

Orders by country rows: 8
+-------+------------+-------------+
|country|total_orders|total_revenue|
+-------+------------+-------------+
|     US|      111231| 176700411.43|
|     JP|       56029|  89046260.07|
|     IN|       56036|  88979795.85|
|     AU|       55992|  88467013.60|
|     GB|       56430|  88415365.08|
|     FR|       55523|  87047234.06|
|     DE|       55047|  86340537.33|
|     CA|       53712|  84199863.64|
+-------+------------+-------------+




#### Write Gold Aggregates to Delta Tables

This cell defines a helper function to write DataFrames to Delta tables in the gold schema:

- `write_to_gold(df, table_name)`: Overwrites the specified gold table with the provided DataFrame using Delta format.
- Writes the following gold-level aggregates:
  - `daily_sales`
  - `top_customers`
  - `revenue_by_category`
  - `orders_by_country`
- Prints status messages for each table written and confirms completion.

In [0]:
# Cell 14 - Write to Gold Delta tables
def write_to_gold(df, table_name):
    target_table = f"{GOLD_SCHEMA}.{table_name}"
    print(f"Writing to {target_table} ...")
    (df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
    print(f"✅ Done: {target_table}")

write_to_gold(df_daily_sales, "daily_sales")
write_to_gold(df_top_customers, "top_customers")
write_to_gold(df_revenue_by_category, "revenue_by_category")
write_to_gold(df_orders_by_country, "orders_by_country")

print("\n✅ All Gold tables written!")

Writing to workspace.retailpulse_gold.daily_sales ...
✅ Done: workspace.retailpulse_gold.daily_sales
Writing to workspace.retailpulse_gold.top_customers ...
✅ Done: workspace.retailpulse_gold.top_customers
Writing to workspace.retailpulse_gold.revenue_by_category ...
✅ Done: workspace.retailpulse_gold.revenue_by_category
Writing to workspace.retailpulse_gold.orders_by_country ...
✅ Done: workspace.retailpulse_gold.orders_by_country

✅ All Gold tables written!



#### Verify Gold Tables

This cell verifies that the gold-level aggregate tables were written successfully:

- Iterates through each gold table (`daily_sales`, `top_customers`, `revenue_by_category`, `orders_by_country`)
- Loads each table from the gold schema
- Prints the row count and number of columns for each table
- Displays a sample of 3 rows from each table for verification

In [0]:
# Cell 16 - Verify Gold tables
for table in ["daily_sales", "top_customers", "revenue_by_category", "orders_by_country"]:
    df = spark.table(f"{GOLD_SCHEMA}.{table}")
    print(f"{GOLD_SCHEMA}.{table}: {df.count():,} rows | {len(df.columns)} columns")
    display(df.head(3))

workspace.retailpulse_gold.daily_sales: 1,827 rows | 3 columns


order_date,total_orders,total_revenue
2020-01-01,276,411922.710000000000000000
2020-01-02,253,412737.050000000000000000
2020-01-03,302,557098.340000000000000000


workspace.retailpulse_gold.top_customers: 50,000 rows | 6 columns


customer_id,total_orders,lifetime_value,name,city,country
8283,17,94762.700000000000000000,Lucas Allen,Vancouver,CA
29629,10,77858.180000000000000000,Lucas Lewis,Houston,US
30365,19,72471.260000000000000000,Elijah Perez,Birmingham,GB


workspace.retailpulse_gold.revenue_by_category: 6 rows | 3 columns


category,total_orders,total_revenue
Electronics,79822,443995204.500000000000000000
Home & Kitchen,85191,122109022.140000000000000000
Sports,84508,94486921.140000000000000000


workspace.retailpulse_gold.orders_by_country: 8 rows | 3 columns


country,total_orders,total_revenue
US,111231,176700411.430000000000000000
JP,56029,89046260.070000000000000000
IN,56036,88979795.850000000000000000
